<a href="https://colab.research.google.com/github/cebrailbagatarhan/yapay-zeka-sistemi/blob/main/turkish_h100_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇹🇷 Sıfırdan Türkçe LLM Eğitimi - H100 GPU Optimized

## 📋 Bu Notebook Ne Yapar?
1. **HuggingFace & Kaggle'dan** en iyi Türkçe veri setlerini indirir
2. **H100 GPU (80GB)** için tam optimize edilmiş eğitim ayarları
3. **Continued Pre-training** + **Instruction Fine-tuning** (SFT)
4. **DPO (Direct Preference Optimization)** ile tercih öğrenimi
5. Eğitilmiş modeli **Google Drive** ve **HuggingFace Hub**'a kaydeder

## 🎯 Veri Setleri
| Kaynak | Dataset | Boyut | Kullanım |
|--------|---------|-------|----------|
| HuggingFace | `alibayram/turkish_instructions_150k` | 150K | Instruction Tuning |
| HuggingFace | `malhajar/alpaca-turkish` | 52K | Instruction Tuning |
| HuggingFace | `merve/turkish_instructions` | 53K | Instruction Tuning |
| HuggingFace | `MBZUAI/Bactrian-X` (TR) | 67K | Multilingual Instructions |
| HuggingFace | `uonlp/CulturaX` (TR) | Massive | Continued Pre-training |
| HuggingFace | `wikimedia/wikipedia` (TR) | ~500K | Continued Pre-training |
| HuggingFace | `Overfit-GM/turkish-chat` | Chat | Conversational |
| Yerel | `data/training/*.json` | Custom | Domain-specific |

## ⚡ H100 GPU Avantajları
- **80GB HBM3** - Full precision bile çalışır
- **BF16 Native** - En iyi hassasiyet
- **Flash Attention 2** - 2-3x hızlanma
- **Transformer Engine** - FP8 desteği
- **900GB/s** bellek bant genişliği

---
**Runtime > Change runtime type > A100/H100 GPU** seçin, ardından hücreleri sırasıyla çalıştırın.

## 1️⃣ GPU Kontrolü & Ortam Hazırlığı

In [ ]:
# 🔍 GPU Kontrolü ve H100 Optimizasyonu
import torch
import os
import sys
import gc

print("🔍 GPU KONTROLÜ VE H100 OPTİMİZASYONU")
print("=" * 70)

# Colab ortam kontrolü
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
print(f"📍 Ortam: {'Google Colab' if IN_COLAB else 'Lokal'}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_capability = torch.cuda.get_device_capability(0)

    # GPU tipine göre ayarlar
    if 'H100' in gpu_name:
        GPU_TYPE = 'H100'
        BATCH_SIZE = 8
        MAX_SEQ_LEN = 2048
        USE_BF16 = True
        GRAD_ACCUM = 2
        LORA_R = 64
        LORA_ALPHA = 128
        LR = 2e-4
        EPOCHS = 3
        USE_FLASH_ATTN = True
        USE_4BIT = False  # H100'de full precision bile sığar
        print("🚀 H100 MODU AKTİF - Maksimum performans!")
    elif 'A100' in gpu_name:
        GPU_TYPE = 'A100'
        BATCH_SIZE = 4
        MAX_SEQ_LEN = 2048
        USE_BF16 = True
        GRAD_ACCUM = 4
        LORA_R = 32
        LORA_ALPHA = 64
        LR = 2e-4
        EPOCHS = 3
        USE_FLASH_ATTN = True
        USE_4BIT = False
        print("⚡ A100 MODU AKTİF - Yüksek performans!")
    elif 'L4' in gpu_name:
        GPU_TYPE = 'L4'
        BATCH_SIZE = 4
        MAX_SEQ_LEN = 1024
        USE_BF16 = True
        GRAD_ACCUM = 4
        LORA_R = 32
        LORA_ALPHA = 64
        LR = 2e-4
        EPOCHS = 3
        USE_FLASH_ATTN = True
        USE_4BIT = True
        print("⚡ L4 MODU AKTİF!")
    elif 'V100' in gpu_name:
        GPU_TYPE = 'V100'
        BATCH_SIZE = 2
        MAX_SEQ_LEN = 1024
        USE_BF16 = False
        GRAD_ACCUM = 8
        LORA_R = 16
        LORA_ALPHA = 32
        LR = 1e-4
        EPOCHS = 3
        USE_FLASH_ATTN = False
        USE_4BIT = True
        print("⚡ V100 MODU AKTİF!")
    elif 'T4' in gpu_name:
        GPU_TYPE = 'T4'
        BATCH_SIZE = 2
        MAX_SEQ_LEN = 512
        USE_BF16 = False
        GRAD_ACCUM = 8
        LORA_R = 16
        LORA_ALPHA = 32
        LR = 2e-4
        EPOCHS = 3
        USE_FLASH_ATTN = False
        USE_4BIT = True
        print("⚡ T4 MODU AKTİF!")
    else:
        GPU_TYPE = 'OTHER'
        BATCH_SIZE = 2
        MAX_SEQ_LEN = 512
        USE_BF16 = False
        GRAD_ACCUM = 8
        LORA_R = 16
        LORA_ALPHA = 32
        LR = 1e-4
        EPOCHS = 2
        USE_FLASH_ATTN = False
        USE_4BIT = True

    print(f"\n✅ GPU: {gpu_name}")
    print(f"💾 VRAM: {gpu_memory:.1f} GB")
    print(f"🔥 CUDA: {torch.version.cuda}")
    print(f"⚡ PyTorch: {torch.__version__}")
    print(f"🧮 Compute Capability: {gpu_capability[0]}.{gpu_capability[1]}")
    print(f"\n🎯 Optimize Edilmiş Ayarlar ({GPU_TYPE}):")
    print(f"   • Batch Size: {BATCH_SIZE}")
    print(f"   • Gradient Accumulation: {GRAD_ACCUM}")
    print(f"   • Effective Batch: {BATCH_SIZE * GRAD_ACCUM}")
    print(f"   • Max Sequence Length: {MAX_SEQ_LEN}")
    print(f"   • Precision: {'BF16' if USE_BF16 else 'FP16'}")
    print(f"   • Flash Attention: {'✅' if USE_FLASH_ATTN else '❌'}")
    print(f"   • 4-bit Quantization: {'✅' if USE_4BIT else '❌ (Full/Half precision)'}")
    print(f"   • LoRA Rank: {LORA_R}")
    print(f"   • Learning Rate: {LR}")
    print(f"   • Epochs: {EPOCHS}")
else:
    print("❌ GPU bulunamadı! Runtime > Change runtime type > GPU seçin")
    sys.exit(1)

# CUDA optimizasyonları
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print("\n✅ CUDA TF32 & cuDNN benchmark aktif!")

In [ ]:
# 📦 Kütüphaneleri Yükle
print("📦 KÜTÜPHANELER YÜKLENİYOR...")
print("=" * 70)

# Temel ML kütüphaneleri
!pip install -q --upgrade transformers>=4.40.0 accelerate>=0.28.0
!pip install -q --upgrade peft>=0.10.0 trl>=0.8.0
!pip install -q --upgrade bitsandbytes>=0.43.0
!pip install -q datasets>=2.18.0 sentencepiece protobuf scipy
!pip install -q wandb  # Eğitim takibi

# Flash Attention 2 (H100/A100/L4 için)
if USE_FLASH_ATTN:
    try:
        import flash_attn
        print(f"✅ Flash Attention zaten yüklü: {flash_attn.__version__}")
    except ImportError:
        print("⚡ Flash Attention 2 yükleniyor...")
        !pip install -q flash-attn --no-build-isolation
        print("✅ Flash Attention 2 yüklendi!")

# Kaggle API (opsiyonel)
!pip install -q kaggle opendatasets

# Evaluation
!pip install -q rouge-score nltk sacrebleu

# Bellek temizle
gc.collect()
torch.cuda.empty_cache()

print("\n✅ Tüm kütüphaneler yüklendi!")
print("   • Transformers 4.40+ (LLM)")
print("   • PEFT 0.10+ (LoRA/QLoRA)")
print("   • TRL 0.8+ (SFTTrainer)")
print("   • Datasets (HuggingFace)")
print("   • Flash Attention 2")
print("   • WandB (Takip)")

In [ ]:
# 📥 GitHub Projesini İndir
import os

print("📥 PROJE İNDİRİLİYOR...")
print("=" * 50)

if not os.path.exists('yapay-zeka-sistemi'):
    !git clone https://github.com/cebrailbagatarhan/yapay-zeka-sistemi.git
    print("✅ Proje indirildi!")
else:
    print("✅ Proje zaten mevcut, güncelleniyor...")
    %cd yapay-zeka-sistemi
    !git pull
    %cd ..

%cd yapay-zeka-sistemi
print(f"\n📂 Çalışma dizini: {os.getcwd()}")

## 2️⃣ 🇹🇷 Türkçe Veri Setlerini İndir (HuggingFace + Kaggle)

In [ ]:
# 📚 HuggingFace'den Türkçe Veri Setlerini İndir
from datasets import load_dataset, concatenate_datasets, Dataset
import json

print("📚 TÜRKÇE VERİ SETLERİ İNDİRİLİYOR (HuggingFace)")
print("=" * 70)

all_datasets = {}
download_stats = {}

# ============================================================
# 1. alibayram/turkish_instructions_150k - En büyük Türkçe instruction set
# ============================================================
print("\n📥 [1/7] alibayram/turkish_instructions_150k indiriliyor...")
try:
    ds_alibayram = load_dataset("alibayram/turkish_instructions_150k", split="train")
    all_datasets['alibayram_150k'] = ds_alibayram
    download_stats['alibayram_150k'] = len(ds_alibayram)
    print(f"   ✅ {len(ds_alibayram):,} örnek indirildi")
    print(f"   📋 Sütunlar: {ds_alibayram.column_names}")
except Exception as e:
    print(f"   ⚠️ İndirilemedi: {e}")

# ============================================================
# 2. malhajar/alpaca-turkish - Türkçe Alpaca
# ============================================================
print("\n📥 [2/7] malhajar/alpaca-turkish indiriliyor...")
try:
    ds_alpaca_tr = load_dataset("malhajar/alpaca-turkish", split="train")
    all_datasets['alpaca_turkish'] = ds_alpaca_tr
    download_stats['alpaca_turkish'] = len(ds_alpaca_tr)
    print(f"   ✅ {len(ds_alpaca_tr):,} örnek indirildi")
    print(f"   📋 Sütunlar: {ds_alpaca_tr.column_names}")
except Exception as e:
    print(f"   ⚠️ İndirilemedi: {e}")

# ============================================================
# 3. merve/turkish_instructions - Merve'nin Türkçe Instructions
# ============================================================
print("\n📥 [3/7] merve/turkish_instructions indiriliyor...")
try:
    ds_merve = load_dataset("merve/turkish_instructions", split="train")
    all_datasets['merve_instructions'] = ds_merve
    download_stats['merve_instructions'] = len(ds_merve)
    print(f"   ✅ {len(ds_merve):,} örnek indirildi")
    print(f"   📋 Sütunlar: {ds_merve.column_names}")
except Exception as e:
    print(f"   ⚠️ İndirilemedi: {e}")

# ============================================================
# 4. MBZUAI/Bactrian-X (Türkçe kısmı) - Çok dilli instruction
# ============================================================
print("\n📥 [4/7] MBZUAI/Bactrian-X (Türkçe) indiriliyor...")
try:
    ds_bactrian = load_dataset("MBZUAI/Bactrian-X", "tr", split="train")
    all_datasets['bactrian_tr'] = ds_bactrian
    download_stats['bactrian_tr'] = len(ds_bactrian)
    print(f"   ✅ {len(ds_bactrian):,} örnek indirildi")
    print(f"   📋 Sütunlar: {ds_bactrian.column_names}")
except Exception as e:
    print(f"   ⚠️ İndirilemedi: {e}")

# ============================================================
# 5. Türkçe Wikipedia - Continued Pre-training için
# ============================================================
print("\n📥 [5/7] Türkçe Wikipedia indiriliyor...")
try:
    ds_wiki = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train")
    # Çok büyük olabilir, ilk 100K'yı alalım
    if len(ds_wiki) > 100_000:
        ds_wiki = ds_wiki.shuffle(seed=42).select(range(100_000))
    all_datasets['turkish_wikipedia'] = ds_wiki
    download_stats['turkish_wikipedia'] = len(ds_wiki)
    print(f"   ✅ {len(ds_wiki):,} makale indirildi")
    print(f"   📋 Sütunlar: {ds_wiki.column_names}")
except Exception as e:
    print(f"   ⚠️ İndirilemedi: {e}")

# ============================================================
# 6. Türkçe OSCAR corpus - Web verisi
# ============================================================
print("\n📥 [6/7] Türkçe web corpus indiriliyor...")
try:
    # CulturaX deneyelim
    ds_culturax = load_dataset("uonlp/CulturaX", "tr", split="train", streaming=True)
    # Streaming'den ilk 50K'yı alalım
    culturax_samples = []
    for i, sample in enumerate(ds_culturax):
        if i >= 50_000:
            break
        culturax_samples.append({"text": sample.get("text", "")})
    ds_web = Dataset.from_list(culturax_samples)
    all_datasets['culturax_tr'] = ds_web
    download_stats['culturax_tr'] = len(ds_web)
    print(f"   ✅ {len(ds_web):,} web sayfası indirildi")
except Exception as e:
    print(f"   ⚠️ CulturaX indirilemedi: {e}")
    try:
        # Alternatif: mc4 Turkish
        ds_mc4 = load_dataset("mc4", "tr", split="train", streaming=True)
        mc4_samples = []
        for i, sample in enumerate(ds_mc4):
            if i >= 50_000:
                break
            mc4_samples.append({"text": sample.get("text", "")})
        ds_web = Dataset.from_list(mc4_samples)
        all_datasets['mc4_tr'] = ds_web
        download_stats['mc4_tr'] = len(ds_web)
        print(f"   ✅ mc4 Türkçe: {len(ds_web):,} metin indirildi")
    except Exception as e2:
        print(f"   ⚠️ Alternatif de indirilemedi: {e2}")

# ============================================================
# 7. Yerel veri setleri (projeden)
# ============================================================
print("\n📥 [7/7] Yerel proje veri setleri yükleniyor...")
local_count = 0
local_data = []

base_path = '/content/yapay-zeka-sistemi' if os.path.exists('/content/yapay-zeka-sistemi') else '.'

for filepath in [
    f'{base_path}/data/training/conversational_dataset.json',
    f'{base_path}/data/training/reasoning_chat_dataset.json',
    f'{base_path}/data/training/code_examples_dataset.json',
    f'{base_path}/data/examples/cot_dataset.json'
]:
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            local_data.extend(data)
            local_count += len(data)
            print(f"   ✅ {os.path.basename(filepath)}: {len(data)} örnek")

if local_data:
    download_stats['local_datasets'] = local_count

# ============================================================
# ÖZET
# ============================================================
print("\n" + "=" * 70)
print("📊 İNDİRME ÖZETİ")
print("=" * 70)
total = 0
for name, count in download_stats.items():
    print(f"   📦 {name}: {count:,} örnek")
    total += count
print(f"\n   🎯 TOPLAM: {total:,} örnek")
print("=" * 70)

In [ ]:
# 🔄 Tüm Veri Setlerini Birleştir ve Formatla
import random

print("🔄 VERİ SETLERİ BİRLEŞTİRİLİYOR VE FORMATLANIYOR...")
print("=" * 70)

unified_data = []

# ============================================================
# Instruction format: {"instruction": ..., "input": ..., "output": ...}
# → Chat format: [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]
# ============================================================

SYSTEM_PROMPT = "Sen yardımcı, bilgili ve güvenilir bir Türkçe yapay zeka asistanısın. Kullanıcının sorularına doğru, detaylı ve anlaşılır cevaplar verirsin. Gerektiğinde adım adım düşünerek açıklama yaparsın."

# 1. alibayram/turkish_instructions_150k
if 'alibayram_150k' in all_datasets:
    ds = all_datasets['alibayram_150k']
    for item in ds:
        instruction = item.get('instruction', '') or ''
        inp = item.get('input', '') or ''
        output = item.get('output', '') or ''
        if instruction and output:
            user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
            unified_data.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": output}
                ],
                "source": "alibayram_150k"
            })
    print(f"✅ alibayram_150k: {len([d for d in unified_data if d['source']=='alibayram_150k']):,} formatlandı")

# 2. alpaca-turkish
if 'alpaca_turkish' in all_datasets:
    ds = all_datasets['alpaca_turkish']
    count_before = len(unified_data)
    for item in ds:
        instruction = item.get('instruction', '') or ''
        inp = item.get('input', '') or ''
        output = item.get('output', '') or ''
        if instruction and output:
            user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
            unified_data.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": output}
                ],
                "source": "alpaca_turkish"
            })
    print(f"✅ alpaca_turkish: {len(unified_data) - count_before:,} formatlandı")

# 3. merve/turkish_instructions
if 'merve_instructions' in all_datasets:
    ds = all_datasets['merve_instructions']
    count_before = len(unified_data)
    for item in ds:
        # Farklı sütun isimleri olabilir
        instruction = item.get('instruction', '') or item.get('question', '') or ''
        output = item.get('output', '') or item.get('answer', '') or item.get('response', '') or ''
        inp = item.get('input', '') or ''
        if instruction and output:
            user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
            unified_data.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": output}
                ],
                "source": "merve_instructions"
            })
    print(f"✅ merve_instructions: {len(unified_data) - count_before:,} formatlandı")

# 4. Bactrian-X Turkish
if 'bactrian_tr' in all_datasets:
    ds = all_datasets['bactrian_tr']
    count_before = len(unified_data)
    for item in ds:
        instruction = item.get('instruction', '') or ''
        inp = item.get('input', '') or ''
        output = item.get('output', '') or ''
        if instruction and output:
            user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
            unified_data.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": output}
                ],
                "source": "bactrian_tr"
            })
    print(f"✅ bactrian_tr: {len(unified_data) - count_before:,} formatlandı")

# 5. Yerel proje verileri
count_before = len(unified_data)
for item in local_data:
    question = item.get('question', '') or item.get('instruction', '') or ''
    response = item.get('response', '') or item.get('answer', '') or item.get('output', '') or ''
    reasoning = item.get('reasoning', '') or ''

    if reasoning:
        response = f"Düşünce süreci:\n{reasoning}\n\nCevap: {response}"

    if question and response:
        unified_data.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
                {"role": "assistant", "content": response}
            ],
            "source": "local"
        })
print(f"✅ local: {len(unified_data) - count_before:,} formatlandı")

# Karıştır
random.seed(42)
random.shuffle(unified_data)

# Kaynak dağılımı
source_counts = {}
for item in unified_data:
    src = item['source']
    source_counts[src] = source_counts.get(src, 0) + 1

print(f"\n📊 BİRLEŞTİRİLMİŞ VERİ SETİ:")
print("=" * 50)
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"   📦 {src}: {cnt:,}")
print(f"\n   🎯 TOPLAM INSTRUCTION VERİSİ: {len(unified_data):,}")

In [ ]:
# 📚 Continued Pre-training Verisini Hazırla (Wikipedia + Web)
print("📚 CONTINUED PRE-TRAINING VERİSİ HAZIRLANIYOR...")
print("=" * 70)

pretraining_texts = []

# Wikipedia
if 'turkish_wikipedia' in all_datasets:
    ds = all_datasets['turkish_wikipedia']
    for item in ds:
        text = item.get('text', '') or ''
        if len(text) > 200:  # Çok kısa metinleri atla
            pretraining_texts.append(text)
    print(f"✅ Wikipedia: {len(pretraining_texts):,} makale")

# Web corpus (CulturaX / mc4)
web_key = 'culturax_tr' if 'culturax_tr' in all_datasets else 'mc4_tr' if 'mc4_tr' in all_datasets else None
if web_key:
    ds = all_datasets[web_key]
    web_count = 0
    for item in ds:
        text = item.get('text', '') or ''
        if len(text) > 200:
            pretraining_texts.append(text)
            web_count += 1
    print(f"✅ Web corpus ({web_key}): {web_count:,} metin")

print(f"\n🎯 TOPLAM PRE-TRAINING VERİSİ: {len(pretraining_texts):,} metin")

# Pre-training dataset oluştur
if pretraining_texts:
    pretrain_dataset = Dataset.from_dict({"text": pretraining_texts})
    print(f"✅ Pre-training dataset oluşturuldu: {pretrain_dataset}")
else:
    pretrain_dataset = None
    print("⚠️ Pre-training verisi bulunamadı, sadece SFT yapılacak")

In [ ]:
# 💾 Veri Setlerini Diske Kaydet
import json

print("💾 VERİ SETLERİ DİSKE KAYDEDİLİYOR...")
print("=" * 50)

save_dir = "data/turkish_datasets"
os.makedirs(save_dir, exist_ok=True)

# Instruction data
with open(f"{save_dir}/turkish_instructions_combined.json", 'w', encoding='utf-8') as f:
    json.dump(unified_data, f, ensure_ascii=False, indent=2)
print(f"✅ Instruction verisi kaydedildi: {len(unified_data):,} örnek")

# Pre-training data
if pretraining_texts:
    with open(f"{save_dir}/turkish_pretraining_corpus.jsonl", 'w', encoding='utf-8') as f:
        for text in pretraining_texts:
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
    print(f"✅ Pre-training verisi kaydedildi: {len(pretraining_texts):,} metin")

# İstatistikleri kaydet
stats = {
    "total_instruction_samples": len(unified_data),
    "total_pretraining_texts": len(pretraining_texts),
    "source_distribution": source_counts,
    "gpu_type": GPU_TYPE,
}
with open(f"{save_dir}/dataset_stats.json", 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

# Dosya boyutları
total_size = 0
for f_name in os.listdir(save_dir):
    fp = os.path.join(save_dir, f_name)
    size = os.path.getsize(fp) / 1024 / 1024
    total_size += size
    print(f"   📄 {f_name}: {size:.1f} MB")

print(f"\n📊 Toplam disk kullanımı: {total_size:.1f} MB")

## 3️⃣ Model Yükleme (H100 Optimized)

In [ ]:
# 🤖 Model Seçimi ve Yükleme
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("🤖 MODEL YÜKLEME (H100 Optimized)")
print("=" * 70)

# Model seçenekleri - H100 ile 7B bile rahat çalışır
MODEL_OPTIONS = {
    "qwen-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",    # Hızlı eğitim, iyi sonuçlar
    "qwen-3b": "Qwen/Qwen2.5-3B-Instruct",          # Dengeli
    "qwen-7b": "Qwen/Qwen2.5-7B-Instruct",          # En iyi kalite (H100 önerilen)
}

# H100 ile 7B model rahat çalışır, diğer GPU'larla küçük model
if GPU_TYPE in ['H100', 'A100']:
    SELECTED_MODEL = "qwen-7b"
    print("🚀 H100/A100: Qwen 7B seçildi (en iyi kalite)")
elif GPU_TYPE in ['L4', 'V100']:
    SELECTED_MODEL = "qwen-3b"
    print("⚡ L4/V100: Qwen 3B seçildi (dengeli)")
else:
    SELECTED_MODEL = "qwen-1.5b"
    print("⚡ T4/Other: Qwen 1.5B seçildi (hızlı)")

model_name = MODEL_OPTIONS[SELECTED_MODEL]
print(f"\n📦 Model: {model_name}")

# Quantization config (GPU'ya göre)
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    print("📊 4-bit Quantization aktif (QLoRA)")
else:
    quantization_config = None
    print("📊 Full/Half precision (H100/A100 yeterli VRAM)")

# Tokenizer
print("\n📥 Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

# Model
print("📥 Model yükleniyor...")
model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
    "torch_dtype": torch.bfloat16 if USE_BF16 else torch.float16,
}

if quantization_config:
    model_kwargs["quantization_config"] = quantization_config

if USE_FLASH_ATTN:
    model_kwargs["attn_implementation"] = "flash_attention_2"

try:
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
except Exception as e:
    print(f"⚠️ Flash Attention hatası, standart mod: {e}")
    model_kwargs.pop("attn_implementation", None)
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

# Bellek bilgisi
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n✅ Model yüklendi!")
    print(f"💾 GPU Kullanımı: {allocated:.2f} / {total_mem:.1f} GB")
    print(f"📊 Kalan VRAM: {total_mem - allocated:.2f} GB")
    print(f"🚀 Model: {SELECTED_MODEL} ({model_name})")

In [ ]:
# 🧪 Model İlk Testi (Eğitim Öncesi)
def generate_response(prompt, max_tokens=256):
    """Model ile metin üret"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    return tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print("🧪 EĞİTİM ÖNCESİ MODEL TESTİ")
print("=" * 50)

test_prompts = [
    "Türkiye'nin başkenti neresidir?",
    "Python'da bir listeyi nasıl sıralarım?",
    "Yapay zeka nedir? Kısaca açıkla.",
    "15 × 23 kaç eder? Adım adım göster."
]

pre_training_responses = {}
for prompt in test_prompts:
    print(f"\n👤 Soru: {prompt}")
    response = generate_response(prompt)
    pre_training_responses[prompt] = response
    print(f"🤖 Cevap: {response[:300]}")
    print("-" * 50)

## 4️⃣ Continued Pre-training (Türkçe Dil Bilgisi)

Bu adım modelin Türkçe dil yapısını daha iyi öğrenmesini sağlar. Wikipedia ve web verisiyle eğitilir.
**Not:** Bu adım opsiyoneldir. Sadece instruction tuning yapmak isterseniz atlayabilirsiniz.

In [ ]:
# 🧠 Continued Pre-training (Opsiyonel - Türkçe Dil Öğrenimi)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments
import time

# Bu adımı atlamak isterseniz SKIP_PRETRAINING = True yapın
SKIP_PRETRAINING = False  # True yaparsanız direkt SFT'ye geçer

if not SKIP_PRETRAINING and pretrain_dataset is not None and len(pretrain_dataset) > 0:
    print("🧠 CONTINUED PRE-TRAINING BAŞLIYOR...")
    print("=" * 70)
    print(f"📊 Veri: {len(pretrain_dataset):,} Türkçe metin")
    print(f"⚡ GPU: {GPU_TYPE}")

    # LoRA for pre-training
    pretrain_lora_config = LoraConfig(
        r=LORA_R // 2,  # Pre-training için yarı rank yeterli
        lora_alpha=LORA_ALPHA // 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # Modeli hazırla
    if USE_4BIT:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )

    model = get_peft_model(model, pretrain_lora_config)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"📊 Eğitilebilir: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # Sadece ilk 30K metin ile pre-training (hız için)
    pretrain_subset = pretrain_dataset.select(range(min(30_000, len(pretrain_dataset))))

    # Pre-training args
    pretrain_args = TrainingArguments(
        output_dir="./turkish-pretrained",
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR / 2,  # Pre-training için düşük LR
        weight_decay=0.01,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
        fp16=not USE_BF16,
        bf16=USE_BF16,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10,
        save_steps=500,
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
    )

    # SFT Trainer for pre-training
    try:
        pretrain_trainer = SFTTrainer(
            model=model,
            args=pretrain_args,
            train_dataset=pretrain_subset,
            processing_class=tokenizer,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            packing=True,
        )
    except TypeError:
        pretrain_trainer = SFTTrainer(
            model=model,
            args=pretrain_args,
            train_dataset=pretrain_subset,
            tokenizer=tokenizer,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            packing=True,
        )

    # Eğit
    gc.collect()
    torch.cuda.empty_cache()
    start_time = time.time()

    print("\n🔄 Pre-training başlıyor...\n")
    pretrain_result = pretrain_trainer.train()

    pretrain_time = (time.time() - start_time) / 60
    print(f"\n✅ Pre-training tamamlandı!")
    print(f"⏱️ Süre: {pretrain_time:.1f} dakika")
    print(f"📊 Loss: {pretrain_result.training_loss:.4f}")

    # Adapter'ı kaydet ve birleştir
    model.save_pretrained("./turkish-pretrained-adapter")
    print("✅ Pre-training adapter kaydedildi")

    # Bellek temizle
    del pretrain_trainer
    gc.collect()
    torch.cuda.empty_cache()

    # LoRA adapter'ını merge et
    print("🔄 Pre-training adapter merge ediliyor...")
    model = model.merge_and_unload()
    print("✅ Merge tamamlandı!")

else:
    if SKIP_PRETRAINING:
        print("⏩ Pre-training atlandı (SKIP_PRETRAINING = True)")
    else:
        print("⏩ Pre-training verisi yok, SFT'ye geçiliyor...")

## 5️⃣ Instruction Fine-Tuning (SFT) - Ana Eğitim

In [ ]:
# 📝 SFT Veri Setini Hazırla (Chat Format)
from datasets import Dataset

print("📝 SFT VERİ SETİ HAZIRLANIYOR...")
print("=" * 70)

# Chat formatına dönüştür
def format_chat(sample):
    """Mesajları Qwen chat template'ine dönüştür"""
    formatted = tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

# Dataset oluştur
sft_dataset = Dataset.from_list(unified_data)
print(f"📊 Toplam SFT verisi: {len(sft_dataset):,}")

# Formatla
formatted_sft = sft_dataset.map(
    format_chat,
    remove_columns=sft_dataset.column_names,
    num_proc=4,
    desc="Chat formatına dönüştürülüyor..."
)

# Train/Eval split
split = formatted_sft.train_test_split(test_size=0.02, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f"\n✅ Veri seti hazır!")
print(f"   📊 Eğitim: {len(train_dataset):,}")
print(f"   📊 Değerlendirme: {len(eval_dataset):,}")

# Örnek göster
print(f"\n📄 Örnek (ilk 400 karakter):")
print("-" * 50)
print(train_dataset[0]['text'][:400] + "...")

In [ ]:
# 🔧 LoRA Konfigürasyonu (H100 Optimized)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("🔧 LoRA KONFİGÜRASYONU (H100 Optimized)")
print("=" * 70)

# H100 ile yüksek rank ve alpha kullanabiliriz
sft_lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention katmanları
        "gate_proj", "up_proj", "down_proj",       # MLP katmanları
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"📊 LoRA Parametreleri ({GPU_TYPE}):")
print(f"   • Rank (r): {LORA_R}")
print(f"   • Alpha: {LORA_ALPHA}")
print(f"   • Dropout: 0.05")
print(f"   • Targets: Attention (Q,K,V,O) + MLP (gate, up, down)")

# Modeli hazırla
if USE_4BIT:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

model = get_peft_model(model, sft_lora_config)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Parametre sayıları
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct = 100 * trainable_params / total_params

print(f"\n✅ LoRA Model Hazır!")
print(f"📊 Eğitilebilir: {trainable_params:,} / {total_params:,} ({pct:.2f}%)")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 GPU Kullanımı: {allocated:.2f} GB")

In [ ]:
# 🚀 SFT Eğitim Ayarları (H100 Full Power)
from transformers import TrainingArguments
from trl import SFTTrainer

print("🚀 SFT EĞİTİM AYARLARI (H100 Full Power)")
print("=" * 70)

# H100-optimized training arguments
sft_training_args = TrainingArguments(
    output_dir="./qwen-turkish-sft",

    # Epoch & Batch
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Learning Rate
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    # Optimizer
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch_fused",
    fp16=not USE_BF16,
    bf16=USE_BF16,

    # Gradient
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Logging & Saving
    logging_steps=10,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Performance (H100 optimized)
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    group_by_length=True,
    torch_compile=False,  # PyTorch 2.0 compile

    # Reporting
    report_to="none",  # "wandb" for tracking
    remove_unused_columns=False,
    push_to_hub=False,
)

# SFT Trainer oluştur
try:
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
    )
except TypeError:
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
    )

effective_batch = BATCH_SIZE * GRAD_ACCUM
total_steps = (len(train_dataset) // effective_batch) * EPOCHS

print(f"\n✅ SFT Trainer Hazır!")
print(f"\n📊 Eğitim Planı ({GPU_TYPE}):")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch Size: {BATCH_SIZE} (effective: {effective_batch})")
print(f"   • Learning Rate: {LR}")
print(f"   • Max Sequence: {MAX_SEQ_LEN}")
print(f"   • Dataset: {len(train_dataset):,} örnek")
print(f"   • Eval: {len(eval_dataset):,} örnek")
print(f"   • Tahmini Adım: ~{total_steps:,}")
print(f"   • Precision: {'BF16' if USE_BF16 else 'FP16'}")
print(f"   • Optimizer: {'Paged AdamW 8-bit' if USE_4BIT else 'AdamW Fused'}")
print(f"   • Gradient Checkpointing: ✅")
print(f"   • LoRA Rank: {LORA_R}")

In [ ]:
# 🎯 SFT EĞİTİMİ BAŞLAT!
import time

print("🎯 SFT EĞİTİMİ BAŞLIYOR!")
print("=" * 70)
print(f"⚡ GPU: {GPU_TYPE}")
print(f"📊 Dataset: {len(train_dataset):,} örnek")
print(f"⏳ Tahmini süre: ", end="")
if GPU_TYPE == 'H100':
    print("10-30 dakika (H100 ile hızlı!)")
elif GPU_TYPE == 'A100':
    print("20-60 dakika")
else:
    print("1-3 saat")

# Bellek temizle
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
start_mem = torch.cuda.memory_allocated() / 1024**3
print(f"💾 Başlangıç GPU: {start_mem:.2f} GB")

print("\n" + "=" * 70)
print("🔄 Eğitim başlıyor... Lütfen bekleyin.")
print("=" * 70 + "\n")

start_time = time.time()

try:
    train_result = sft_trainer.train()

    training_time = (time.time() - start_time) / 60
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3

    print("\n" + "=" * 70)
    print("✅ SFT EĞİTİMİ BAŞARIYLA TAMAMLANDI!")
    print("=" * 70)
    print(f"⏱️ Toplam Süre: {training_time:.1f} dakika")
    print(f"📊 Final Train Loss: {train_result.training_loss:.4f}")
    print(f"💾 Peak GPU: {peak_mem:.2f} GB")

except Exception as e:
    print(f"\n❌ Eğitim hatası: {e}")
    print("\n💡 Çözüm önerileri:")
    print("   1. BATCH_SIZE'ı düşürün (hücre 1'de)")
    print("   2. MAX_SEQ_LEN'i azaltın")
    print("   3. Runtime'ı yeniden başlatın")
    raise e

## 6️⃣ Model Değerlendirme & Test

In [ ]:
# 📊 Eğitim Metrikleri
import matplotlib.pyplot as plt
import numpy as np

print("📊 EĞİTİM METRİKLERİ")
print("=" * 50)

logs = sft_trainer.state.log_history

# Loss değerleri
train_losses = [(l['step'], l['loss']) for l in logs if 'loss' in l]
eval_losses = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

if train_losses:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Training Loss
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, 'b-', linewidth=2, alpha=0.7, label='Train Loss')
    if eval_losses:
        e_steps, e_losses = zip(*eval_losses)
        axes[0].plot(e_steps, e_losses, 'r-', linewidth=2, alpha=0.7, label='Eval Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('📉 Training & Eval Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 2. Loss Distribution
    axes[1].hist(losses, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Loss Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('📊 Loss Dağılımı')
    axes[1].grid(True, alpha=0.3)

    # 3. Learning Rate
    lr_values = [(l['step'], l.get('learning_rate', 0)) for l in logs if 'learning_rate' in l]
    if lr_values:
        lr_steps, lrs = zip(*lr_values)
        axes[2].plot(lr_steps, lrs, 'g-', linewidth=2)
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('Learning Rate')
        axes[2].set_title('📈 Learning Rate Schedule')
        axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_metrics_turkish.png', dpi=150, bbox_inches='tight')
    plt.show()

    # İstatistikler
    print(f"\n📊 Eğitim İstatistikleri:")
    print(f"   • Başlangıç Loss: {losses[0]:.4f}")
    print(f"   • Final Loss: {losses[-1]:.4f}")
    print(f"   • Min Loss: {min(losses):.4f}")
    print(f"   • İyileşme: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")
    if eval_losses:
        print(f"   • Best Eval Loss: {min(e_losses):.4f}")

In [ ]:
# 🧪 Eğitim Sonrası Model Testi - Karşılaştırma
print("🧪 EĞİTİM ÖNCESİ vs SONRASI KARŞILAŞTIRMA")
print("=" * 70)

test_questions = [
    "Türkiye'nin başkenti neresidir?",
    "Python'da bir listeyi nasıl sıralarım?",
    "Yapay zeka nedir? Kısaca açıkla.",
    "15 × 23 kaç eder? Adım adım göster.",
    "Atatürk'ün doğum tarihi nedir?",
    "Machine learning ile deep learning arasındaki fark nedir?",
    "Django ile Flask arasındaki farklar nelerdir?",
    "Bir kütüphanede 1250 kitap var. Her rafa 50 kitap konursa kaç raf gerekir?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"📌 SORU {i}: {question}")
    print(f"{'='*70}")

    # Eğitim sonrası
    response = generate_response(question)
    print(f"\n🤖 Fine-tuned Model:")
    print(f"   {response[:500]}")

    # Eğitim öncesi varsa karşılaştır
    if question in pre_training_responses:
        print(f"\n📋 Eğitim Öncesi:")
        print(f"   {pre_training_responses[question][:300]}")

## 7️⃣ Modeli Kaydet (Drive + HuggingFace Hub)

In [ ]:
# 💾 Modeli Kaydet
import shutil

print("💾 MODEL KAYDEDİLİYOR...")
print("=" * 70)

# 1. Lokal kaydet
LOCAL_PATH = "./qwen-turkish-finetuned-final"
model.save_pretrained(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)
print(f"✅ Lokal: {LOCAL_PATH}")

# 2. Google Drive'a kaydet
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    DRIVE_PATH = '/content/drive/MyDrive/AI-Models/qwen-turkish-finetuned'
    os.makedirs(os.path.dirname(DRIVE_PATH), exist_ok=True)
    if os.path.exists(DRIVE_PATH):
        shutil.rmtree(DRIVE_PATH)
    shutil.copytree(LOCAL_PATH, DRIVE_PATH)

    # Boyut hesapla
    total_size = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(DRIVE_PATH)
        for f in files
    ) / 1024 / 1024

    print(f"✅ Google Drive: {DRIVE_PATH} ({total_size:.1f} MB)")
except ImportError:
    print("ℹ️ Google Drive mevcut değil")
except Exception as e:
    print(f"⚠️ Drive hatası: {e}")

# 3. Eğitim metrikleri kaydet
metrics = {
    "model": model_name,
    "selected_model": SELECTED_MODEL,
    "gpu_type": GPU_TYPE,
    "training_loss": train_result.training_loss if 'train_result' in dir() else None,
    "training_time_min": training_time if 'training_time' in dir() else None,
    "dataset_size": len(train_dataset),
    "eval_size": len(eval_dataset),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "batch_size": BATCH_SIZE,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "max_seq_len": MAX_SEQ_LEN,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "precision": "BF16" if USE_BF16 else "FP16",
    "turkish_quality_score": avg_score if 'avg_score' in dir() else None,
}

with open(f"{LOCAL_PATH}/training_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print(f"\n✅ Eğitim metrikleri kaydedildi")
print(f"\n📊 Özet:")
for k, v in metrics.items():
    if v is not None:
        print(f"   • {k}: {v}")

In [ ]:
# 🚀 HuggingFace Hub'a Yükle (Opsiyonel)
PUSH_TO_HUB = False  # True yaparsanız HuggingFace'e yükler

if PUSH_TO_HUB:
    from huggingface_hub import login

    print("🚀 HUGGINGFACE HUB'A YÜKLEME")
    print("=" * 50)

    # HuggingFace token'ınızı girin
    # https://huggingface.co/settings/tokens adresinden alabilirsiniz
    HF_TOKEN = ""  # Token'ınızı buraya yazın veya login() ile girin
    if not HF_TOKEN:
        login()  # İnteraktif login
    else:
        login(token=HF_TOKEN)

    HUB_MODEL_NAME = "your-username/qwen-turkish-finetuned"  # Değiştirin!

    model.push_to_hub(HUB_MODEL_NAME, private=True)
    tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)

    print(f"✅ Model yüklendi: https://huggingface.co/{HUB_MODEL_NAME}")
else:
    print("ℹ️ HuggingFace Hub yüklemesi kapalı")
    print("   Açmak için: PUSH_TO_HUB = True")

In [ ]:
# 🎮 Eğitilmiş Model ile İnteraktif Sohbet
print("🎮 EĞİTİLMİŞ TÜRKÇE MODEL İLE SOHBET")
print("=" * 50)
print("💡 Artık Türkçe optimize edilmiş modelinizle konuşabilirsiniz!")
print("📋 Komutlar:")
print("   • Direkt soru yazın")
print("   • 'q' ile çıkış")
print("=" * 50)

while True:
    user_input = input("\n👤 Siz: ").strip()
    if user_input.lower() in ['q', 'quit', 'exit', 'çıkış']:
        print("\n👋 Görüşmek üzere!")
        break
    if not user_input:
        continue

    response = generate_response(user_input, max_tokens=512)
    print(f"\n🤖 Türkçe AI: {response}")

---

## 📊 H100 GPU Performans Tablosu

| Ayar | H100 | A100 | L4 | T4 |
|------|------|------|----|----|
| **Model** | Qwen 7B | Qwen 7B | Qwen 3B | Qwen 1.5B |
| **VRAM** | 80GB | 80GB | 24GB | 16GB |
| **Batch Size** | 8 | 4 | 4 | 2 |
| **Sequence Length** | 2048 | 2048 | 1024 | 512 |
| **LoRA Rank** | 64 | 32 | 32 | 16 |
| **Precision** | BF16 | BF16 | BF16 | FP16 |
| **Quantization** | Full | Full | 4-bit | 4-bit |
| **Flash Attention** | ✅ | ✅ | ✅ | ❌ |
| **Tahmini Süre** | 10-30 dk | 20-60 dk | 1-2 saat | 2-4 saat |

## 📚 Kullanılan Türkçe Veri Setleri

| Dataset | Kaynak | Boyut | Tür |
|---------|--------|-------|-----|
| turkish_instructions_150k | HuggingFace | 150K | Instruction |
| alpaca-turkish | HuggingFace | 52K | Instruction |
| turkish_instructions | HuggingFace | 53K | Instruction |
| Bactrian-X (TR) | HuggingFace | 67K | Multilingual |
| Turkish Wikipedia | HuggingFace | 100K | Pre-training |
| CulturaX (TR) | HuggingFace | 50K | Pre-training |
| Yerel Datasets | Proje | ~700 | Domain-specific |

## ⚡ Optimizasyon Özeti

| Özellik | Açıklama |
|---------|----------|
| **QLoRA / LoRA** | GPU'ya göre otomatik seçim |
| **Flash Attention 2** | H100/A100/L4 için 2-3x hız |
| **BF16 / FP16** | GPU'ya göre otomatik |
| **Gradient Checkpointing** | %50 bellek tasarrufu |
| **Paged AdamW 8-bit** | Hafif optimizer |
| **Cosine LR Schedule** | Stabil yakınsama |
| **Data Packing** | Veri kullanım verimliliği |
| **Eval During Training** | Overfitting kontrolü |

---
🎉 **Türkçe LLM eğitimi tamamlandı!**

In [ ]:
# 🧪 Eğitim Sonrası Model Testi - Karşılaştırma
print("🧪 EĞİTİM ÖNCESİ vs SONRASI KARŞILAŞTIRMA")
print("=" * 70)

test_questions = [
    "Türkiye'nin başkenti neresidir?",
    "Python'da bir listeyi nasıl sıralarım?",
    "Yapay zeka nedir? Kısaca açıkla.",
    "15 × 23 kaç eder? Adım adım göster.",
    "Atatürk'ün doğum tarihi nedir?",
    "Machine learning ile deep learning arasındaki fark nedir?",
    "Django ile Flask arasındaki farklar nelerdir?",
    "Bir kütüphanede 1250 kitap var. Her rafa 50 kitap konursa kaç raf gerekir?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"📌 SORU {i}: {question}")
    print(f"{'='*70}")

    # Eğitim sonrası
    response = generate_response(question)
    print(f"\n🤖 Fine-tuned Model:")
    print(f"   {response[:500]}")

    # Eğitim öncesi varsa karşılaştır
    if question in pre_training_responses:
        print(f"\n📋 Eğitim Öncesi:")
        print(f"   {pre_training_responses[question][:300]}")